## 03 - Cleaning

Detects the language of each review, cleans and lemmatizes the text and adds date columns.
It also exports the spaCy stopword list that the dashboard uses.

Input: `data/processed/trenord_reviews_anon.csv`

Output: `data/processed/trenord_reviews_clean.csv` and `dashboard/stopwords_it.txt`

In [1]:
# --- IMPORTS ---
import re

import pandas as pd
import spacy
from spacy.lang.it.stop_words import STOP_WORDS as italian_stopwords
from lingua import Language, LanguageDetectorBuilder

# --- LOAD ANONYMISED DATA ---
# We load the CSV saved by 02_anonymization.ipynb (hashed review_id, no author).
# The raw file is never read here.
df = pd.read_csv("../data/processed/trenord_reviews_anon.csv")

print(f"Loaded {len(df)} reviews")
print(f"Columns: {df.columns.tolist()}")

Loaded 7329 reviews
Columns: ['review_id', 'rating', 'text', 'thumbs_up', 'app_version', 'date', 'has_reply', 'reply_text', 'reply_date']


In [2]:
# --- STEP 1: ADD DATE COLUMNS ---
# The "date" column is already a datetime object, but we extract
# individual components to make time-based analysis easier later.

# Make sure the date column is parsed as datetime, just to be safe
df["date"] = pd.to_datetime(df["date"])

# Extract individual date components as separate columns
df["year"]       = df["date"].dt.year        # e.g. 2024
df["month"]      = df["date"].dt.month       # e.g. 3 (March)
df["day_of_week"]= df["date"].dt.day_name()  # e.g. "Monday"
df["year_month"] = df["date"].dt.to_period("M").astype(str) # e.g. "2024-03"

# Quick check - print the date range of our reviews
print(f"Date range: {df['date'].min().date()} → {df['date'].max().date()}")
print(f"\nSample of new columns:")
print(df[["date", "year", "month", "day_of_week", "year_month"]].head())

Date range: 2015-05-05 → 2026-08-04

Sample of new columns:
                 date  year  month day_of_week year_month
0 2026-08-04 16:01:12  2026      8     Tuesday    2026-08
1 2026-08-03 17:29:22  2026      8      Monday    2026-08
2 2026-08-03 06:48:29  2026      8      Monday    2026-08
3 2026-08-02 22:42:05  2026      8      Sunday    2026-08
4 2026-07-31 21:30:14  2026      7      Friday    2026-07


In [3]:
# --- STEP 2: DETECT LANGUAGE ---
# We use a two-level approach based on manual inspection of the data:
#
# - If lingua detects Italian (any confidence) → accept as Italian.
#   Short texts like "pessima" or "ottima" are correctly identified as Italian
#   but with low confidence - we found 921 such cases during inspection.
#
# - Any other language → require confidence >= 0.5, else label as "unknown".

detector = LanguageDetectorBuilder.from_all_languages().build()

CONFIDENCE_THRESHOLD = 0.5

# Quick look at what the detector returns for a single text:
# a list of (language, confidence) pairs ranked by confidence (text not printed).
sample_text = df["text"].iloc[0]
print("Confidence values:")
for cv in detector.compute_language_confidence_values(sample_text)[:5]:
    print(f"  {cv.language.iso_code_639_1.name.lower():<10} {cv.value:.4f}")


def detect_language(text):
    if not isinstance(text, str) or not text.strip():
        return "unknown"

    # Ranked list of (language, confidence) - empty if text is too short to analyze
    confidence_values = detector.compute_language_confidence_values(text)
    if not confidence_values:
        return "unknown"

    top = confidence_values[0]
    lang = top.language.iso_code_639_1.name.lower()
    confidence = top.value

    # Italian is accepted at any confidence (see note above)
    if lang == "it":
        return "it"

    return lang if confidence >= CONFIDENCE_THRESHOLD else "unknown"


print("\nDetecting languages...")
df["language"] = df["text"].apply(detect_language)

print("\nLanguage distribution:")
print(df["language"].value_counts())

unknown_count = (df["language"] == "unknown").sum()
print(f"\n{unknown_count} reviews labeled as 'unknown' - excluded from topic modeling.")

Confidence values:
  it         1.0000
  la         0.0000
  en         0.0000
  pt         0.0000
  tl         0.0000

Detecting languages...

Language distribution:
language
it         6951
unknown     363
en            5
la            5
fr            2
es            1
mi            1
pt            1
Name: count, dtype: int64

363 reviews labeled as 'unknown' - excluded from topic modeling.


In [4]:
# --- STEP 3: CLEAN TEXT ---
# We create a new column "text_clean" with normalized text for NLP analysis.

def clean_text(text):
    
    # Guard clause: if the value is not a string (e.g. NaN), return empty string
    if not isinstance(text, str):
        return ""
    
    # Step 1: lowercase everything so "Treno" and "treno" are the same word
    text = text.lower()
    
    # Step 2: remove URLs (e.g. "http://..." or "www....")
    text = re.sub(r"http\S+|www\.\S+", "", text)
    
    # Step 3: remove everything that is not a letter or a space.
    # [^a-zàèéìíòóùú\s] means "anything that is NOT a-z, an Italian accented vowel, or a space"
    # This removes numbers, punctuation, emojis, and special characters
    text = re.sub(r"[^a-zàèéìíòóùú\s]", " ", text)
    
    # Step 4: collapse multiple spaces into one and remove leading/trailing spaces
    # \s+ means "one or more whitespace characters"
    text = re.sub(r"\s+", " ", text).strip()
    
    return text

# Apply the function to every row in the "text" column
df["text_clean"] = df["text"].apply(clean_text)

# Add word count on the ORIGINAL text
# This is useful later to filter out very short reviews that carry little information
df["word_count"] = df["text"].apply(lambda x: len(str(x).split()))

# Sanity check - counts only (review texts are never printed)
empty_after_cleaning = (df["text_clean"] == "").sum()
print(f"Reviews cleaned         : {len(df):,}")
print(f"Empty after cleaning    : {empty_after_cleaning:,}")
print(f"Average words (original): {df['word_count'].mean():.1f}")

Reviews cleaned         : 7,329
Empty after cleaning    : 10
Average words (original): 19.9


In [5]:
# --- STEP 3b: DUPLICATE CHECK ON THE ITALIAN REVIEWS ---
# Topic modeling works on the Italian reviews, so the check is done on them.
# It needs text_clean, so it comes after the cleaning step. Counts only, no texts.
italian_reviews = df[df["language"] == "it"]

n_rows = len(italian_reviews)
n_distinct_ids = italian_reviews["review_id"].nunique()
n_distinct_texts = italian_reviews["text_clean"].nunique()

# value_counts() gives how many reviews have each cleaned text.
# A text counted more than once is shared by several reviews: each such text is one group.
text_counts = italian_reviews["text_clean"].value_counts()
shared_texts = text_counts[text_counts > 1]
n_shared_groups = len(shared_texts)
n_reviews_sharing_text = shared_texts.sum()

# Length of the reviews that share their text, compared with all Italian reviews
is_shared = italian_reviews["text_clean"].isin(shared_texts.index)
median_words_shared = italian_reviews.loc[is_shared, "word_count"].median()
median_words_all = italian_reviews["word_count"].median()

print(f"Italian reviews                       : {n_rows:,}")
print(f"Distinct review_id                    : {n_distinct_ids:,}")
print(f"Distinct text_clean                   : {n_distinct_texts:,}")
print(f"Reviews sharing their text_clean      : {n_reviews_sharing_text:,}")
print(f"Groups of reviews with the same text  : {n_shared_groups:,}")
print(f"Median words, reviews sharing text    : {median_words_shared:.0f}")
print(f"Median words, all Italian reviews     : {median_words_all:.0f}")

Italian reviews                       : 6,951
Distinct review_id                    : 6,951
Distinct text_clean                   : 6,341
Reviews sharing their text_clean      : 743
Groups of reviews with the same text  : 133
Median words, reviews sharing text    : 2
Median words, all Italian reviews     : 14


### Duplicate check

Every Italian review has its own `review_id`: 6,951 rows and 6,951 distinct ids, so no review was collected twice.

The cleaned texts are a different story. 743 reviews share their `text_clean` with at least one other review, in 133 groups. They are short: their median length is 2 words, against 14 for all Italian reviews, so many users write them in the same words. They are kept: each one comes from a different review, with its own rating and date, so it is a separate opinion and not a copy.

In [6]:
# --- STEP 4: STOPWORDS REMOVAL AND LEMMATIZATION ---
# We use spacy with the Italian model "it_core_news_lg".
# spacy processes each review and for every word (token) it gives us:
# - token.lemma_    : the base form of the word
# - token.is_stop   : True if the word is a stopword
# - token.is_alpha  : True if the word contains only letters (no numbers or punctuation)
# We keep only words that are: not a stopword AND only letters AND at least 3 characters long

# Load the Italian language model
# This model knows Italian grammar and can lemmatize Italian words
nlp = spacy.load("it_core_news_lg")

def lemmatize_text(text):
    # If the text is empty, return empty string
    if not isinstance(text, str) or len(text.strip()) == 0:
        return ""
    
    # spacy processes the text and returns a Doc object
    # A Doc is a sequence of Token objects, one per word
    doc = nlp(text)
    
    # For each token we check three conditions:
    # 1. token.is_alpha  → keep only words made of letters (no punctuation leftovers)
    # 2. not token.is_stop → remove stopwords like "il", "di", "che", "è"
    # 3. len(token.lemma_) > 2 → remove very short words like "va" or "fa"
    # If all three conditions are true, we keep the lemmatized form of the word
    lemmas = [
        token.lemma_                # the base form e.g. "acquistare" instead of "acquistato"
        for token in doc
        if token.is_alpha           # only letters
        and not token.is_stop       # not a stopword
        and len(token.lemma_) > 2   # at least 3 characters
    ]
    
    # Join the list of lemmas back into a single string
    return " ".join(lemmas)

# Apply to the cleaned text column
# This may take a few minutes on 7000+ reviews
print("Lemmatizing reviews... this may take a few minutes.")
df["text_lemmatized"] = df["text_clean"].apply(lemmatize_text)

# Sanity check - counts only (review texts are never printed)
empty_after_lemmatization = (df["text_lemmatized"] == "").sum()
print(f"Empty after lemmatization: {empty_after_lemmatization:,}")

Lemmatizing reviews... this may take a few minutes.
Empty after lemmatization: 135


In [7]:
# --- STEP 5: EXPORT THE STOPWORD LIST FOR THE DASHBOARD ---
# The dashboard must use exactly the same Italian stopwords as this pipeline, without depending on spaCy.
# We write the spaCy list to a static text file: one word per line, sorted, UTF-8, with a header comment.
STOPWORDS_FILE = "../dashboard/stopwords_it.txt"

with open(STOPWORDS_FILE, "w", encoding="utf-8", newline="\n") as f:
    f.write(f"# Generated by notebooks/03_cleaning.ipynb from spaCy {spacy.__version__} STOP_WORDS (Italian). "
            "Do not edit by hand.\n")
    for word in sorted(italian_stopwords):
        f.write(word + "\n")

print(f"Stopwords written: {len(italian_stopwords)}")

Stopwords written: 624


In [8]:
# --- STEP 6: SAVE PROCESSED DATA ---
# We save the cleaned and enriched dataframe to a new CSV file.
# This is the file that all subsequent notebooks will load.
# We never overwrite the raw file - we always create a new processed version.

df.to_csv("../data/processed/trenord_reviews_clean.csv", index=False, encoding="utf-8-sig")

print(f"Processed data saved!")
print(f"Rows    : {len(df)}")
print(f"Columns : {df.columns.tolist()}")

Processed data saved!
Rows    : 7329
Columns : ['review_id', 'rating', 'text', 'thumbs_up', 'app_version', 'date', 'has_reply', 'reply_text', 'reply_date', 'year', 'month', 'day_of_week', 'year_month', 'language', 'text_clean', 'word_count', 'text_lemmatized']
